[User Browser]
                                     │
                                     ▼
                            (Ngrok Public URL)
                                     │
                                     ▼
                           [FastAPI Server / UI]
                                     │
                                     ▼
                     ┌─────── [ORCHESTRATOR] ────────┐
                     │                               │
                1. Check                        6. Log & Action
                     │                               │
                     ▼                               ▼
             [Semantic Cache]               [Automated Actions]
               (ChromaDB)                  (Email, SQLite Stats)
                     │
          2. If Miss (Run Agents)
                     │
                     ▼
    ┌──────────────────────────────────────────────────┐
    │                 MULTI-AGENT LOOP                 │
    │                                                  │
    │  1. [Intent Agent] ──▶ Extracts track & skills   │
    │           │                                      │
    │           ├─▶ [RAG Agent] (ChromaDB + SQLite)    │
    │           │          (Runs in Parallel)          │
    │           ├─▶ [Web Agent] (Tavily API)           │
    │           │                                      │
    │           ▼                                      │
    │  2. [Recommendation Agent] ──▶ Drafts Roadmap    │
    │           │                                      │
    │           ▼                                      │
    │  3. [Critic Agent] ──▶ Reviews & Fixes           │
    │           │                                      │
    │           ▼                                      │
    │  4. [Final Response Agent] ──▶ Formats Markdown  │
    └──────────────────────────────────────────────────┘

Setup, Secrets & LLM Cascade

In [1]:
# ==========================================
# CELL 1: SETUP, SECRETS & LLM CASCADE
# ==========================================
import os

# 1. Installs FIRST (Prevents PyTorch circular import error in Colab)
print("Installing dependencies... (This takes about 1-2 minutes)")
!pip install -q fastapi uvicorn pyngrok pydantic sqlalchemy pandas pypdf sentence-transformers chromadb requests accelerate bitsandbytes transformers

# 2. Imports after PIP install
import sys
import json
import time
import requests
import re
import torch
from pydantic import BaseModel

# 3. GPU Check
HAS_GPU = torch.cuda.is_available()
print(f"GPU Status: {'Enabled (' + torch.cuda.get_device_name(0) + ')' if HAS_GPU else 'CPU Only (Will skip local HF models)'}")

# 4. Directories Setup
BASE_DIR = "/content/darby"
DIRS = [
    os.path.join(BASE_DIR, "data/documents"),
    os.path.join(BASE_DIR, "data/db"),
    os.path.join(BASE_DIR, "data/chroma")
]
for d in DIRS:
    os.makedirs(d, exist_ok=True)
print(f"Created directories under {BASE_DIR}")

# 5. Secrets Loader (Colab Userdata & Fallback)
def get_secret(key_name, default=""):
    try:
        from google.colab import userdata
        val = userdata.get(key_name)
        if val:
            return val
    except Exception:
        pass
    return os.environ.get(key_name, default)

# Retrieve keys from Colab Secrets (Add secrets named GEMINI_API_KEY, TAVILY_API_KEY, NGROK_AUTHTOKEN, HF_TOKEN)
GEMINI_API_KEY = get_secret("GEMINI_API_KEY")
TAVILY_API_KEY = get_secret("TAVILY_API_KEY")
NGROK_AUTHTOKEN = get_secret("NGROK_AUTHTOKEN")
HF_TOKEN = get_secret("HF_TOKEN")

# Set Default Env Vars
os.environ["LLM_MODEL"] = "gemini-2.5-flash"
os.environ["EMBEDDING_MODEL"] = "all-MiniLM-L6-v2"
os.environ["DATABASE_URL"] = f"sqlite:///{os.path.join(BASE_DIR, 'data/db/darby.db')}"

# 6. LLM Provider Infrastructure
class LLMProvider:
    def generate_text(self, prompt: str) -> str:
        raise NotImplementedError
    def generate_structured(self, prompt: str, schema: BaseModel) -> BaseModel:
        raise NotImplementedError

class DemoProvider(LLMProvider):
    def generate_text(self, prompt: str) -> str:
        return "Demo Mode: The system gracefully degraded. API keys or GPU memory exhausted. (Simulated Response)"

    def generate_structured(self, prompt: str, schema: BaseModel) -> BaseModel:
        if "Intent" in schema.__name__:
            return schema.model_validate({"intent": "general", "target_tracks": [], "experience_level": "beginner", "known_skills": [], "goals": [], "needs_rag": True, "needs_web_research": False, "language": "en"})
        return schema.model_validate({"verdict": "approved", "issues": [], "required_fixes": []})

class GeminiRESTProvider(LLMProvider):
    def __init__(self, api_key: str, model="gemini-2.5-flash"):
        self.api_key = api_key
        self.model = model
        self.url = f"https://generativelanguage.googleapis.com/v1beta/models/{self.model}:generateContent?key={self.api_key}"
        self.fallback = DemoProvider()

    def _call_api(self, payload: dict, retries=2) -> dict:
        headers = {"Content-Type": "application/json"}
        for attempt in range(retries + 1):
            resp = requests.post(self.url, json=payload, headers=headers, timeout=15)
            if resp.status_code == 200:
                return resp.json()
            elif resp.status_code == 429:
                print(f"[Gemini] 429 Rate Limit. Backing off... (Attempt {attempt+1})")
                time.sleep(2 ** attempt)
            else:
                resp.raise_for_status()
        raise Exception("Rate limit or connection exhausted.")

    def generate_text(self, prompt: str) -> str:
        try:
            payload = {"contents": [{"parts": [{"text": prompt}]}]}
            data = self._call_api(payload)
            return data["candidates"][0]["content"]["parts"][0]["text"]
        except Exception as e:
            print(f"[Gemini Fallback] Error: {e}")
            return self.fallback.generate_text(prompt)

    def generate_structured(self, prompt: str, schema: BaseModel) -> BaseModel:
        try:
            payload = {
                "contents": [{"parts": [{"text": prompt + "\n\nRespond ONLY with valid JSON."}]}],
                "generationConfig": {"response_mime_type": "application/json"}
            }
            data = self._call_api(payload)
            json_str = data["candidates"][0]["content"]["parts"][0]["text"]
            return schema.model_validate_json(json_str)
        except Exception as e:
            print(f"[Gemini Structured Fallback] Error: {e}")
            return self.fallback.generate_structured(prompt, schema)

class HFLocalProvider(LLMProvider):
    def __init__(self, fallback_provider):
        self.fallback = fallback_provider
        self.model_name = "Qwen/Qwen2.5-3B-Instruct"
        print(f"[HFLocal] Loading {self.model_name} in 4-bit...")
        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16
        )
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name, token=HF_TOKEN if HF_TOKEN else None)
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_name,
            quantization_config=bnb_config,
            device_map="auto",
            token=HF_TOKEN if HF_TOKEN else None
        )
        print("[HFLocal] Model loaded successfully.")

    def generate_text(self, prompt: str) -> str:
        try:
            messages = [{"role": "user", "content": prompt}]
            text = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inputs = self.tokenizer([text], return_tensors="pt").to("cuda")
            outputs = self.model.generate(**inputs, max_new_tokens=1024, do_sample=False)
            response = self.tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
            return response.split("assistant\n")[-1].strip()
        except Exception as e:
            print(f"[HFLocal Fallback] Error: {e}")
            return self.fallback.generate_text(prompt)

    def generate_structured(self, prompt: str, schema: BaseModel) -> BaseModel:
        try:
            instruction = f"{prompt}\n\nReturn strictly valid JSON matching this schema: {schema.model_json_schema()}"
            text_resp = self.generate_text(instruction)
            match = re.search(r'\{.*\}', text_resp.replace('\n', ''), re.DOTALL)
            if match:
                return schema.model_validate_json(match.group(0))
            return schema.model_validate_json(text_resp)
        except Exception as e:
            print(f"[HFLocal Structured Fallback] Error: {e}")
            return self.fallback.generate_structured(prompt, schema)

# 7. Global LLM Initializer
LLM_PROVIDER_MODE = "auto"
llm_client = None

def get_llm():
    global llm_client
    if llm_client is not None:
        return llm_client

    demo = DemoProvider()
    gemini = GeminiRESTProvider(GEMINI_API_KEY) if GEMINI_API_KEY else demo

    if LLM_PROVIDER_MODE == "auto":
        if HAS_GPU:
            try:
                llm_client = HFLocalProvider(fallback_provider=gemini)
            except Exception as e:
                print(f"[Auto Init] HF Local failed ({e}). Falling back to Gemini.")
                llm_client = gemini
        else:
            llm_client = gemini
    elif LLM_PROVIDER_MODE == "gemini":
        llm_client = gemini
    elif LLM_PROVIDER_MODE == "hf_local" and HAS_GPU:
        try:
            llm_client = HFLocalProvider(fallback_provider=gemini)
        except Exception as e:
            llm_client = gemini
    else:
        llm_client = demo

    return llm_client

# --- Test Execution ---
print("\nTesting LLM Cascade...")
test_llm = get_llm()
print(f"Active Provider: {test_llm.__class__.__name__}")
response = test_llm.generate_text("Say exactly the word 'OK'.")
print(f"LLM Test Response: {response}")

Installing dependencies... (This takes about 1-2 minutes)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 78.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 10.6 MB/s et

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[HFLocal] Model loaded successfully.
Active Provider: HFLocalProvider
LLM Test Response: OK


SQLite Schema & Data Seeding

In [2]:
# ==========================================
# CELL 2: SQLITE SCHEMA & DATA SEEDING
# ==========================================
import os
import pandas as pd
from datetime import datetime
from sqlalchemy import create_engine, Column, Integer, String, DateTime
from sqlalchemy.orm import declarative_base, sessionmaker

# 1. Database Setup
DB_PATH = "/content/darby/data/db/darby.db"
# check_same_thread=False is required for FastAPI + SQLite concurrency
engine = create_engine(f"sqlite:///{DB_PATH}", connect_args={"check_same_thread": False})
SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)
Base = declarative_base()

# 2. Schema Definitions
class Resource(Base):
    __tablename__ = "resources"
    id = Column(Integer, primary_key=True, index=True)
    title = Column(String, index=True)
    url = Column(String)
    track = Column(String, index=True)
    resource_type = Column(String)
    difficulty = Column(String)

class QueryLog(Base):
    __tablename__ = "query_log"
    id = Column(Integer, primary_key=True, index=True)
    query = Column(String, index=True)
    timestamp = Column(DateTime, default=datetime.utcnow)

class Stat(Base):
    __tablename__ = "stats"
    metric_name = Column(String, primary_key=True, index=True)
    metric_value = Column(Integer, default=0)

# Create tables
Base.metadata.create_all(bind=engine)

# 3. Seeding Markdown Documents (8 Core Tracks)
DOCS_DIR = "/content/darby/data/documents"

docs_content = {
    "machine_learning.md": """# Machine Learning Track
## Prerequisites
Python, Linear Algebra, Calculus, Probability.
## Learning Stages
1. Data Manipulation (NumPy, Pandas)
2. Classical ML (Scikit-Learn, Regression, Classification, Clustering)
3. Deep Learning Basics (PyTorch, Neural Networks)
## Tools
Scikit-Learn, PyTorch, Jupyter.
## Projects
1. House Price Prediction (Linear Regression).
2. Customer Segmentation (K-Means).
## What NOT to learn yet
Transformers or LLMs before understanding basic neural networks.
## Ready for next stage?
You can build and evaluate a Scikit-Learn model on a tabular dataset.
""",
    "data_science.md": """# Data Science Track
## Prerequisites
Python, Basic Statistics.
## Learning Stages
1. Data Cleaning & Wrangling (Pandas, SQL)
2. Exploratory Data Analysis (Matplotlib, Seaborn)
3. Predictive Modeling (Scikit-Learn)
## Tools
Pandas, SQL, Tableau.
## Projects
1. EDA on a Kaggle Dataset.
2. Sales Forecasting.
## What NOT to learn yet
Big Data tools like Hadoop/Spark until you master Pandas and standard SQL.
## Ready for next stage?
You can query a database and visualize trends cleanly.
""",
    "frontend.md": """# Frontend Development Track
## Prerequisites
Basic internet concepts (HTTP, Browsers).
## Learning Stages
1. Core Web (HTML, CSS, JavaScript)
2. Frameworks (React or Vue)
3. Advanced UI (Tailwind, State Management)
## Tools
React, Vite, Tailwind CSS.
## Projects
1. Responsive Weather App (Vanilla JS).
2. To-Do List with Local Storage (React).
## What NOT to learn yet
Next.js or Redux before mastering basic React hooks.
## Ready for next stage?
You can build a multi-component React app that fetches data from an API.
""",
    "backend.md": """# Backend Development Track
## Prerequisites
Programming fundamentals (Variables, Loops, Functions), Command Line basics.
## Learning Stages
1. Web Servers & APIs (FastAPI, Express, or Spring Boot)
2. Databases (PostgreSQL, SQL Syntax)
3. Authentication & Deployment (JWT, Docker)
## Tools
FastAPI, PostgreSQL, Docker.
## Projects
1. CRUD REST API (Blog system).
2. Secure Login/Registration API with JWT.
## What NOT to learn yet
Microservices or Kubernetes before mastering monolithic CRUD APIs.
## Ready for next stage?
You can deploy a secure API connected to a relational database.
""",
    "software_engineering.md": """# Software Engineering Track
## Prerequisites
Object-Oriented Programming (OOP) in any language, Git.
## Learning Stages
1. Data Structures & Algorithms (Arrays, Trees, Graphs, Big-O)
2. System Design Basics (Caching, Load Balancing)
3. Testing & CI/CD (Unit Tests, GitHub Actions)
## Tools
Git, PyTest/JUnit, Redis.
## Projects
1. CI/CD Pipeline for a simple app.
2. Rate Limiter Middleware.
## What NOT to learn yet
Complex cloud architectures (AWS EKS) before learning basic Linux and Docker.
## Ready for next stage?
You can comfortably solve medium LeetCode problems and write unit tests.
""",
    "cybersecurity.md": """# Cybersecurity Track
## Prerequisites
Networking basics (TCP/IP, DNS), Linux Command Line.
## Learning Stages
1. Network Security & Cryptography
2. Web Vulnerabilities (OWASP Top 10, SQLi, XSS)
3. Ethical Hacking / Penetration Testing
## Tools
Wireshark, Kali Linux, Burp Suite.
## Projects
1. Set up a secure home network and firewall.
2. Perform a vulnerability assessment on a purposely vulnerable app (e.g., OWASP Juice Shop).
## What NOT to learn yet
Advanced Exploit Development or Reverse Engineering before mastering basic web pentesting.
## Ready for next stage?
You can identify and explain common web vulnerabilities and patch them.
""",
    "cloud_devops.md": """# Cloud & DevOps Track
## Prerequisites
Linux Administration, Bash Scripting.
## Learning Stages
1. Containerization (Docker)
2. CI/CD Pipelines (Jenkins, GitHub Actions)
3. Infrastructure as Code (Terraform) & Cloud (AWS/Azure)
## Tools
Docker, Terraform, AWS.
## Projects
1. Dockerize a multi-tier application (Frontend + Backend + DB).
2. Write Terraform scripts to provision a basic cloud server.
## What NOT to learn yet
Kubernetes before thoroughly understanding Docker and CI/CD basics.
## Ready for next stage?
You can fully automate the deployment of a simple app from GitHub to a server.
""",
    "mobile.md": """# Mobile Development Track
## Prerequisites
Basic programming concepts.
## Learning Stages
1. UI Fundamentals & Navigation (Flutter/Dart or React Native)
2. State Management & API Integration
3. Device Features (Camera, GPS, Local Storage)
## Tools
Flutter, React Native, Firebase.
## Projects
1. Habit Tracker App with Local Storage.
2. Movie Browsing App fetching from a public API.
## What NOT to learn yet
Native modules (Kotlin/Swift bridging) before mastering the cross-platform framework itself.
## Ready for next stage?
You can build a responsive mobile app with multiple screens and API data.
"""
}

for filename, content in docs_content.items():
    with open(os.path.join(DOCS_DIR, filename), "w", encoding="utf-8") as f:
        f.write(content)
print(f"✅ Seeded {len(docs_content)} Markdown documents into {DOCS_DIR}")

# 4. Seeding CSV Resources
CSV_PATH = "/content/darby/data/resources.csv"
resources_data = [
    {"title": "Python Official Tutorial", "url": "https://docs.python.org/3/tutorial/", "track": "machine-learning", "resource_type": "documentation", "difficulty": "beginner"},
    {"title": "Scikit-Learn Docs", "url": "https://scikit-learn.org/stable/", "track": "machine-learning", "resource_type": "documentation", "difficulty": "intermediate"},
    {"title": "Kaggle", "url": "https://www.kaggle.com/", "track": "data-science", "resource_type": "platform", "difficulty": "intermediate"},
    {"title": "MDN Web Docs", "url": "https://developer.mozilla.org/en-US/", "track": "frontend", "resource_type": "documentation", "difficulty": "beginner"},
    {"title": "React.dev", "url": "https://react.dev/", "track": "frontend", "resource_type": "documentation", "difficulty": "intermediate"},
    {"title": "FastAPI Docs", "url": "https://fastapi.tiangolo.com/", "track": "backend", "resource_type": "documentation", "difficulty": "intermediate"},
    {"title": "PostgreSQL Tutorial", "url": "https://www.postgresqltutorial.com/", "track": "backend", "resource_type": "tutorial", "difficulty": "beginner"},
    {"title": "System Design Primer", "url": "https://github.com/donnemartin/system-design-primer", "track": "software-engineering", "resource_type": "repository", "difficulty": "advanced"},
    {"title": "OWASP Top 10", "url": "https://owasp.org/www-project-top-ten/", "track": "cybersecurity", "resource_type": "documentation", "difficulty": "beginner"},
    {"title": "Docker Curriculum", "url": "https://docker-curriculum.com/", "track": "cloud-devops", "resource_type": "tutorial", "difficulty": "beginner"},
    {"title": "Flutter Documentation", "url": "https://docs.flutter.dev/", "track": "mobile", "resource_type": "documentation", "difficulty": "beginner"}
]
df = pd.DataFrame(resources_data)
df.to_csv(CSV_PATH, index=False)
print(f"✅ Created {CSV_PATH} with {len(resources_data)} resources")

# 5. Initialize SQLite Database Data
db = SessionLocal()

# Clear tables for fresh run (safe idempotency)
db.query(Resource).delete()
db.query(QueryLog).delete()
db.query(Stat).delete()
db.commit()

# Seed Stats
db.add(Stat(metric_name="total_queries", metric_value=0))
db.add(Stat(metric_name="cache_hits", metric_value=0))

# Load Resources into SQLite
for item in resources_data:
    db.add(Resource(**item))
db.commit()
db.close()

print("✅ SQLite database initialized and seeded successfully.")

✅ Seeded 8 Markdown documents into /content/darby/data/documents
✅ Created /content/darby/data/resources.csv with 11 resources
✅ SQLite database initialized and seeded successfully.


RAG Ingestion Pipeline & Semantic Cache

In [3]:
# ==========================================
# CELL 3: RAG INGESTION & SEMANTIC CACHE
# ==========================================
import os
import glob
import uuid
import chromadb
from chromadb.utils import embedding_functions
from pypdf import PdfReader

# 1. Initialize ChromaDB
CHROMA_DIR = "/content/darby/data/chroma"
chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)

# 2. Setup Embedding Function
EMBEDDING_MODEL = os.environ.get("EMBEDDING_MODEL", "all-MiniLM-L6-v2")
print(f"Loading embedding model: {EMBEDDING_MODEL}...")
embedding_func = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=EMBEDDING_MODEL)

# 3. Create Collections
rag_collection = chroma_client.get_or_create_collection(name="darby_knowledge", embedding_function=embedding_func)
cache_collection = chroma_client.get_or_create_collection(name="darby_cache", embedding_function=embedding_func)

# Clear existing RAG data for a fresh ingest (Idempotent)
if rag_collection.count() > 0:
    rag_collection.delete(where={"source_type": {"$in": ["md", "pdf"]}})

# 4. Text Chunking Helper
def chunk_text(text, chunk_size=800, overlap=150):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += (chunk_size - overlap)
    return chunks

# 5. Ingestion Pipeline (Markdown & PDF)
DOCS_DIR = "/content/darby/data/documents"

def extract_text(filepath):
    ext = filepath.split('.')[-1].lower()
    text = ""
    if ext == 'md' or ext == 'txt':
        with open(filepath, 'r', encoding='utf-8') as f:
            text = f.read()
    elif ext == 'pdf':
        try:
            reader = PdfReader(filepath)
            for page in reader.pages:
                extracted = page.extract_text()
                if extracted:
                    text += extracted + "\n"
        except Exception as e:
            print(f"Error reading PDF {filepath}: {e}")
    return text

print("Starting document ingestion...")
files = glob.glob(os.path.join(DOCS_DIR, "*.*"))
total_chunks = 0

for file in files:
    ext = file.split('.')[-1].lower()
    if ext not in ['md', 'txt', 'pdf']:
        continue

    filename = os.path.basename(file)
    content = extract_text(file)

    if not content.strip():
        continue

    chunks = chunk_text(content)

    docs = []
    metadatas = []
    ids = []

    for i, chunk in enumerate(chunks):
        docs.append(chunk)
        metadatas.append({"source": filename, "source_type": ext, "chunk_index": i})
        ids.append(f"{filename}_chunk_{i}_{uuid.uuid4().hex[:6]}")

    if docs:
        rag_collection.add(documents=docs, metadatas=metadatas, ids=ids)
        total_chunks += len(docs)
        print(f" - Ingested {filename}: {len(docs)} chunks")

print(f"✅ Ingestion complete. Total chunks in RAG database: {rag_collection.count()}")
if rag_collection.count() == 0:
    print("❌ WARNING: RAG database is empty! The project requirements state RAG must not be empty.")

# 6. Semantic Cache Functions
def check_semantic_cache(query: str, threshold: float = 0.35):
    """
    Checks if a semantically identical query exists in the cache.
    Threshold: L2 distance (lower is closer). 0.35 is a strict semantic match.
    """
    if cache_collection.count() == 0:
        return None

    results = cache_collection.query(query_texts=[query], n_results=1)

    if results['distances'] and len(results['distances'][0]) > 0:
        dist = results['distances'][0][0]
        if dist < threshold:
            print(f"[Cache] Hit! Distance: {dist:.4f}")
            # The cached answer is stored in the metadata
            return results['metadatas'][0][0].get('answer')

    return None

def save_to_cache(query: str, response: str):
    """
    Saves the query as the embedded document, and the response in the metadata.
    Does NOT cache Demo Mode responses.
    """
    if "Demo Mode:" in response:
        return

    cache_id = f"cache_{uuid.uuid4().hex[:8]}"
    cache_collection.add(
        documents=[query],
        metadatas=[{"answer": response}],
        ids=[cache_id]
    )

Loading embedding model: all-MiniLM-L6-v2...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Starting document ingestion...
 - Ingested cybersecurity.md: 1 chunks
 - Ingested machine_learning.md: 1 chunks
 - Ingested backend.md: 1 chunks
 - Ingested frontend.md: 1 chunks
 - Ingested data_science.md: 1 chunks
 - Ingested cloud_devops.md: 1 chunks
 - Ingested software_engineering.md: 1 chunks
 - Ingested mobile.md: 1 chunks
✅ Ingestion complete. Total chunks in RAG database: 8


Multi-Agent Pipeline

In [4]:
# ==========================================
# CELL 4: MULTI-AGENT PIPELINE
# ==========================================
import json
import requests
from typing import List
from pydantic import BaseModel, Field

# ------------------------------------------
# 1. SCHEMAS
# ------------------------------------------
class IntentOutput(BaseModel):
    intent: str
    target_tracks: List[str]
    experience_level: str
    known_skills: List[str]
    goals: List[str]
    needs_rag: bool
    needs_web_research: bool
    language: str = "en"

class CriticOutput(BaseModel):
    verdict: str # "approved" or "needs_revision"
    issues: List[str]
    required_fixes: List[str]

# ------------------------------------------
# 2. AGENT IMPLEMENTATIONS
# ------------------------------------------

def run_intent_agent(user_query: str, chat_history: str = "") -> IntentOutput:
    llm = get_llm()
    prompt = f"""
    Analyze the following user query for a technology-learning platform.
    Extract the core intent, target tracks (e.g., "machine-learning", "game-dev", "frontend"), current skills, and language.

    CRITICAL RULES:
    1. If the user is asking about what to learn, roadmaps, tracks, or resources, set `needs_rag` to true.
    2. If the user asks for current trends, salaries, or very specific external info, set `needs_web_research` to true.
    3. We support ANY tech track. Extract it accurately even if it's niche.

    Chat History:
    {chat_history}

    User Query: "{user_query}"
    """
    return llm.generate_structured(prompt, IntentOutput)

def run_rag_agent(user_query: str, target_tracks: List[str]) -> str:
    # 1. Fetch text chunks from ChromaDB
    context = "=== INTERNAL KNOWLEDGE BASE (Documents) ===\n"
    if rag_collection.count() > 0:
        results = rag_collection.query(query_texts=[user_query], n_results=4)
        if results['documents'] and len(results['documents'][0]) > 0:
            for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
                context += f"Source [{meta.get('source', 'unknown')}]: {doc}\n\n"
        else:
            context += "No relevant documents found.\n\n"

    # 2. Fetch curated URLs from SQLite database
    context += "=== CURATED RESOURCES (Database Links) ===\n"
    db = SessionLocal()
    try:
        query = db.query(Resource)
        if target_tracks:
            # Simple fuzzy matching for the first target track
            track_filter = target_tracks[0].lower().replace(" ", "-")
            query = query.filter(Resource.track.like(f"%{track_filter}%"))

        resources = query.limit(5).all()
        if resources:
            for r in resources:
                context += f"- [{r.resource_type.upper()}] {r.title}: {r.url} (Diff: {r.difficulty})\n"
        else:
            context += "No curated database links found for this track.\n"
    finally:
        db.close()

    return context

def run_web_research_agent(user_query: str) -> str:
    if not TAVILY_API_KEY:
        return "Web search skipped: TAVILY_API_KEY is missing."

    try:
        search_query = f"{user_query} technology learning roadmap prerequisites resources"
        url = "https://api.tavily.com/search"
        payload = {"api_key": TAVILY_API_KEY, "query": search_query, "search_depth": "basic", "max_results": 3}
        resp = requests.post(url, json=payload, timeout=10)
        resp.raise_for_status()
        results = resp.json().get("results", [])

        context = "=== WEB RESEARCH ===\n"
        for r in results:
            context += f"- {r.get('title')}: {r.get('content')} ({r.get('url')})\n"
        return context
    except Exception as e:
        return f"Web search failed: {str(e)}"

def run_recommendation_agent(user_query: str, intent_data: dict, rag_context: str, web_context: str, previous_draft: str = "") -> str:
    llm = get_llm()
    prompt = f"""
    You are an expert technical mentor. Generate a highly specific, actionable roadmap.
    We support ALL technology tracks.

    USER QUERY: {user_query}
    INTENT / PROFILE: {json.dumps(intent_data)}

    EVIDENCE:
    {rag_context}

    {web_context}

    MANDATORY TEMPLATE STRUCTURE:
    1. Recommended Track: Name it and explain in 1 sentence why it fits.
    2. Current Level: Acknowledge what they already know.
    3. Roadmap Stages: Create 2-3 logical stages (what to learn first, what next). Detail specific topics and tools per stage.
    4. What NOT to Learn Yet: Explicitly name 1-2 advanced topics they should avoid for now.
    5. Resources: Provide 3-8 REAL links ONLY pulled from the RAG CONTEXT or WEB CONTEXT above.
       - Label each link with its source: e.g., `[internal]`, `[web]`, `[model]`.
       - NEVER invent or guess a URL. If you don't have URLs in the context, do not include them.
    6. Starter Projects: Suggest 1-2 concrete projects.
    7. Milestone Criteria: How will the user know they are ready to move to the next stage?
    8. ONE Immediate Next Action: Give exactly ONE highly specific command (e.g., "Complete the first module of the MDN HTML tutorial").
    """

    if previous_draft:
        prompt += f"\n\nCRITIC REJECTION - PREVIOUS DRAFT TO FIX:\n{previous_draft}\n(You MUST apply these required fixes based on the critic's feedback.)"

    return llm.generate_text(prompt)

def run_critic_agent(draft: str) -> CriticOutput:
    llm = get_llm()
    prompt = f"""
    Review this technical roadmap draft. You must enforce strict quality standards.

    DRAFT:
    {draft}

    CRITERIA FOR REJECTION (needs_revision):
    1. The draft is vague and does not contain concrete tools or technologies.
    2. It is missing the explicit "ONE Immediate Next Action".
    3. It includes URLs without a source label like [internal] or [web].
    4. The learning order is illogical (e.g., teaching React before JavaScript).

    If it fails any criteria, output 'needs_revision' and list the EXACT required fixes.
    If it meets all criteria, output 'approved'.
    """
    return llm.generate_structured(prompt, CriticOutput)

def run_final_agent(user_query: str, approved_draft: str, language: str) -> str:
    llm = get_llm()
    prompt = f"""
    You are Darby, an elite technical mentor.
    Format the provided approved draft into beautiful, highly professional Markdown.

    RULES:
    1. Translate the entire response to language: {language}.
    2. Use standard Markdown ONLY (## Headings, - Bullet lists, [Link](url)).
    3. DO NOT use any emojis. This must look like a professional academic/career document.
    4. Hide all internal agent notes, chain-of-thought, or JSON formatting.
    5. Maintain the source labels on links (e.g., `[internal]`, `[web]`).

    APPROVED DRAFT TO FORMAT:
    {approved_draft}
    """
    return llm.generate_text(prompt)

print("✅ Agent Implementations Loaded Successfully.")

✅ Agent Implementations Loaded Successfully.


Orchestrator & Automated Actions

In [5]:
# ==========================================
# CELL 5: ORCHESTRATOR & AUTOMATED ACTIONS
# ==========================================
import asyncio
import time
from datetime import datetime

# ------------------------------------------
# 1. AUTOMATED ACTIONS & LOGGING
# ------------------------------------------
def log_query_and_stats(user_query: str, is_cache_hit: bool):
    """Logs the query to SQLite and updates dashboard stats."""
    db = SessionLocal()
    try:
        # Log query
        db.add(QueryLog(query=user_query))

        # Update Total Queries
        total_stat = db.query(Stat).filter(Stat.metric_name == "total_queries").first()
        if total_stat:
            total_stat.metric_value += 1

        # Update Cache Hits if applicable
        if is_cache_hit:
            cache_stat = db.query(Stat).filter(Stat.metric_name == "cache_hits").first()
            if cache_stat:
                cache_stat.metric_value += 1

        db.commit()
    except Exception as e:
        print(f"DB Logging Error: {e}")
        db.rollback()
    finally:
        db.close()

def simulate_email_sending(email_address: str, markdown_content: str):
    """Simulates the automated action of emailing the roadmap."""
    print(f"\n[{datetime.utcnow().isoformat()}] 📧 SIMULATED EMAIL INITIATED")
    print(f"TO: {email_address}")
    print(f"SUBJECT: Your Personalized Darby Tech Roadmap")
    print("-" * 40)
    # Print the first 200 characters to prove it works without flooding the console
    print(f"{markdown_content[:200]}...\n[EMAIL END]")
    print("-" * 40)
    return True

# ------------------------------------------
# 2. ORCHESTRATOR (ASYNC WORKFLOW)
# ------------------------------------------
async def process_user_query(user_query: str, chat_history: str = "") -> dict:
    trace = []

    def add_trace(agent_name, status, duration_ms):
        trace.append({"agent": agent_name, "status": status, "time_ms": int(duration_ms)})

    # A. Check Semantic Cache First
    start_time = time.time()
    cached_response = check_semantic_cache(user_query, threshold=0.35)
    if cached_response:
        add_trace("Semantic Cache", "CACHE_HIT", (time.time() - start_time) * 1000)
        log_query_and_stats(user_query, is_cache_hit=True)
        return {"response": cached_response, "trace": trace, "cache_hit": True}

    add_trace("Semantic Cache", "MISS", (time.time() - start_time) * 1000)

    # B. Intent & Profile Agent
    start_time = time.time()
    try:
        intent_data = run_intent_agent(user_query, chat_history)
        add_trace("Intent Agent", "SUCCESS", (time.time() - start_time) * 1000)
    except Exception as e:
        add_trace("Intent Agent", f"FAILED: {e}", (time.time() - start_time) * 1000)
        # Fallback intent if it crashes
        intent_data = IntentOutput(intent="unknown", target_tracks=[], experience_level="unknown", known_skills=[], goals=[], needs_rag=True, needs_web_research=False, language="en")

    # C. Run RAG and Web Research IN PARALLEL
    rag_context = "RAG Skipped."
    web_context = "Web Skipped."

    start_time = time.time()
    # We use asyncio.to_thread to run synchronous LLM/DB requests in parallel without blocking
    rag_task = asyncio.to_thread(run_rag_agent, user_query, intent_data.target_tracks) if intent_data.needs_rag else asyncio.to_thread(lambda: "RAG skipped.")
    web_task = asyncio.to_thread(run_web_research_agent, user_query) if intent_data.needs_web_research else asyncio.to_thread(lambda: "Web research skipped.")

    rag_context, web_context = await asyncio.gather(rag_task, web_task)
    add_trace("Parallel RAG & Web", "SUCCESS", (time.time() - start_time) * 1000)

    # D. Recommendation Agent (Drafting)
    start_time = time.time()
    draft = run_recommendation_agent(user_query, intent_data.model_dump(), rag_context, web_context)
    add_trace("Recommendation Agent", "DRAFTED", (time.time() - start_time) * 1000)

    # E. Critic Agent (Review Loop)
    start_time = time.time()
    try:
        critic_result = run_critic_agent(draft)
        if critic_result.verdict.lower() == "needs_revision":
            add_trace("Critic Agent", "REJECTED - REVISING", (time.time() - start_time) * 1000)
            # Run Recommendation again with feedback
            start_time_rev = time.time()
            draft = run_recommendation_agent(user_query, intent_data.model_dump(), rag_context, web_context, previous_draft=draft)
            add_trace("Recommendation Agent", "REVISED", (time.time() - start_time_rev) * 1000)
        else:
            add_trace("Critic Agent", "APPROVED", (time.time() - start_time) * 1000)
    except Exception as e:
        add_trace("Critic Agent", f"SKIPPED (Error: {e})", (time.time() - start_time) * 1000)

    # F. Final Response Agent
    start_time = time.time()
    final_markdown = run_final_agent(user_query, draft, intent_data.language)
    add_trace("Final Response Agent", "FORMATTED", (time.time() - start_time) * 1000)

    # G. Save to Cache & Log Stats
    save_to_cache(user_query, final_markdown)
    log_query_and_stats(user_query, is_cache_hit=False)

    return {"response": final_markdown, "trace": trace, "cache_hit": False}

print("✅ Orchestrator and Automated Actions Loaded Successfully.")

✅ Orchestrator and Automated Actions Loaded Successfully.


In [8]:
import threading
import time
import uvicorn

def run_api():
    uvicorn.run("main:app", host="0.0.0.0", port=8000, log_level="info")

server_thread = threading.Thread(target=run_api, daemon=True)
server_thread.start()
time.sleep(4)
print("STEP 1 DONE: server thread started")

ERROR:    Error loading ASGI app. Could not import module "main".


STEP 1 DONE: server thread started


In [9]:
import threading
import time
import uvicorn

def run_api():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

threading.Thread(target=run_api, daemon=True).start()
time.sleep(5)
print("WAIT: look for a line that says Uvicorn running")

INFO:     Started server process [1957]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


WAIT: look for a line that says Uvicorn running


FastAPI, UI & Ngrok Server

In [11]:
# ==========================================
# CELL 6: FASTAPI, UI & NGROK SERVER
# ==========================================
import nest_asyncio
import uvicorn
import getpass
from fastapi import FastAPI
from fastapi.responses import HTMLResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from pyngrok import ngrok

# 1. FastAPI Setup
app = FastAPI(title="Darby API")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# 2. API Schemas
class ChatRequest(BaseModel):
    message: str
    history: str = ""

class EmailRequest(BaseModel):
    email: str
    markdown: str

# 3. API Routes
@app.post("/api/chat")
async def chat_endpoint(req: ChatRequest):
    result = await process_user_query(req.message, req.history)
    return result

@app.get("/api/stats")
def get_stats():
    db = SessionLocal()
    try:
        t_stat = db.query(Stat).filter(Stat.metric_name == "total_queries").first()
        c_stat = db.query(Stat).filter(Stat.metric_name == "cache_hits").first()
        total = t_stat.metric_value if t_stat else 0
        hits = c_stat.metric_value if c_stat else 0
        return {"total_queries": total, "cache_hits": hits}
    finally:
        db.close()

@app.post("/api/email-roadmap")
def email_endpoint(req: EmailRequest):
    success = simulate_email_sending(req.email, req.markdown)
    return {"status": "sent" if success else "failed"}

# 4. Single-Page HTML UI
HTML_CONTENT = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Darby - Tech Career Mentor</title>
    <script src="https://cdn.tailwindcss.com"></script>
    <script src="https://cdn.jsdelivr.net/npm/marked/marked.min.js"></script>
    <style>
        .markdown-body h1, .markdown-body h2, .markdown-body h3 { font-weight: bold; margin-top: 1rem; margin-bottom: 0.5rem; color: #1f2937; }
        .markdown-body h1 { font-size: 1.5rem; }
        .markdown-body h2 { font-size: 1.25rem; border-bottom: 1px solid #e5e7eb; padding-bottom: 0.25rem; }
        .markdown-body ul { list-style-type: disc; margin-left: 1.5rem; margin-bottom: 1rem; }
        .markdown-body p { margin-bottom: 1rem; line-height: 1.6; }
        .markdown-body a { color: #2563eb; text-decoration: underline; }
        .markdown-body code { background-color: #f3f4f6; padding: 0.2rem 0.4rem; border-radius: 0.25rem; font-size: 0.875em; }
    </style>
</head>
<body class="bg-gray-50 flex flex-col h-screen font-sans">

    <!-- Navbar -->
    <header class="bg-white shadow-sm px-6 py-4 flex justify-between items-center z-10">
        <div class="flex items-center gap-2">
            <div class="bg-blue-600 text-white font-bold rounded-lg p-2 leading-none">D</div>
            <h1 class="text-xl font-bold text-gray-800">Darby</h1>
            <span class="text-sm text-gray-500 hidden sm:inline ml-2">Find your path in tech.</span>
        </div>
        <div class="text-xs font-semibold text-gray-500 bg-gray-100 px-3 py-1 rounded-full" id="stats-badge">
            Queries: 0 | Cache Hits: 0
        </div>
    </header>

    <!-- Chat Container -->
    <main id="chat-container" class="flex-grow overflow-y-auto p-4 sm:p-6 space-y-6 max-w-4xl mx-auto w-full pb-32">
        <div class="flex gap-4">
            <div class="w-8 h-8 rounded-full bg-blue-600 flex-shrink-0 flex items-center justify-center text-white text-sm font-bold">D</div>
            <div class="bg-white border border-gray-200 rounded-2xl rounded-tl-none px-5 py-4 shadow-sm text-gray-800">
                <p>Hello! I'm Darby, your multi-agent tech career mentor.</p>
                <p class="text-sm text-gray-500 mt-2">Try asking: "I want to become a Machine Learning Engineer. I know basic Python. What's next?"</p>
            </div>
        </div>
    </main>

    <!-- Input Area -->
    <footer class="fixed bottom-0 w-full bg-white border-t border-gray-200 p-4">
        <div class="max-w-4xl mx-auto flex gap-2">
            <input type="text" id="user-input" class="flex-grow border border-gray-300 rounded-xl px-4 py-3 focus:outline-none focus:ring-2 focus:ring-blue-500 focus:border-transparent transition-all" placeholder="Tell me your goals or what you want to learn..." onkeypress="handleEnter(event)">
            <button onclick="sendMessage()" id="send-btn" class="bg-blue-600 hover:bg-blue-700 text-white font-medium rounded-xl px-6 py-3 transition-colors flex items-center gap-2">
                <span>Send</span>
            </button>
        </div>
    </footer>

    <script>
        const chatContainer = document.getElementById('chat-container');
        const userInput = document.getElementById('user-input');
        const sendBtn = document.getElementById('send-btn');
        let chatHistory = "";

        async function updateStats() {
            try {
                const res = await fetch('/api/stats');
                const data = await res.json();
                document.getElementById('stats-badge').innerText = `Queries: ${data.total_queries} | Cache Hits: ${data.cache_hits}`;
            } catch (e) { console.error(e); }
        }
        updateStats();

        function handleEnter(e) {
            if (e.key === 'Enter') sendMessage();
        }

        async function sendMessage() {
            const message = userInput.value.trim();
            if (!message) return;

            userInput.value = '';
            userInput.disabled = true;
            sendBtn.disabled = true;
            sendBtn.innerHTML = 'Wait...';

            chatContainer.insertAdjacentHTML('beforeend', `
                <div class="flex gap-4 justify-end">
                    <div class="bg-blue-600 text-white rounded-2xl rounded-tr-none px-5 py-4 shadow-sm max-w-[85%]">
                        <p>${message}</p>
                    </div>
                </div>
            `);
            scrollToBottom();

            const loadingId = 'loading-' + Date.now();
            chatContainer.insertAdjacentHTML('beforeend', `
                <div id="${loadingId}" class="flex gap-4">
                    <div class="w-8 h-8 rounded-full bg-gray-200 flex-shrink-0 animate-pulse"></div>
                    <div class="bg-white border border-gray-200 rounded-2xl rounded-tl-none px-5 py-4 shadow-sm text-gray-500 animate-pulse">
                        Agents are analyzing...
                    </div>
                </div>
            `);
            scrollToBottom();

            try {
                const response = await fetch('/api/chat', {
                    method: 'POST',
                    headers: { 'Content-Type': 'application/json' },
                    body: JSON.stringify({ message: message, history: chatHistory.slice(-1000) })
                });

                const data = await response.json();
                document.getElementById(loadingId).remove();

                chatHistory += `\\nUser: ${message}\\nDarby: ${data.response}`;

                let traceHtml = '<div class="flex flex-wrap gap-2 mb-3">';
                if (data.cache_hit) {
                    traceHtml += `<span class="px-2 py-1 bg-green-100 text-green-700 text-xs font-semibold rounded-md border border-green-200">⚡ Semantic Cache Hit (Bypassed LLM)</span>`;
                }
                data.trace.forEach(t => {
                    let color = t.status.includes('SUCCESS') || t.status.includes('APPROVED') || t.status.includes('FORMATTED') ? 'bg-gray-100 text-gray-700 border-gray-200' :
                                t.status.includes('FAILED') || t.status.includes('REJECTED') ? 'bg-red-100 text-red-700 border-red-200' : 'bg-yellow-100 text-yellow-700 border-yellow-200';
                    traceHtml += `<span class="px-2 py-1 ${color} text-xs rounded-md border" title="Agent: ${t.agent}">⚙️ ${t.agent} (${t.time_ms}ms)</span>`;
                });
                traceHtml += '</div>';

                const markdownHtml = marked.parse(data.response);
                const blockId = 'resp-' + Date.now();

                // Constructing HTML without nested JS templates
                chatContainer.insertAdjacentHTML('beforeend', `
                    <div class="flex gap-4">
                        <div class="w-8 h-8 rounded-full bg-blue-600 flex-shrink-0 flex items-center justify-center text-white text-sm font-bold">D</div>
                        <div class="bg-white border border-gray-200 rounded-2xl rounded-tl-none px-5 py-4 shadow-sm text-gray-800 w-full max-w-3xl">
                            ${traceHtml}
                            <div class="markdown-body">${markdownHtml}</div>

                            <!-- Automated Actions Bar -->
                            <div class="mt-4 pt-4 border-t border-gray-100 flex flex-wrap gap-2" id="${blockId}">
                                <button class="copy-btn text-xs font-medium px-3 py-1.5 border border-gray-300 rounded hover:bg-gray-50">📋 Copy</button>
                                <button class="export-btn text-xs font-medium px-3 py-1.5 border border-gray-300 rounded hover:bg-gray-50">💾 Export .md</button>
                                <button class="email-btn text-xs font-medium px-3 py-1.5 bg-gray-900 text-white rounded hover:bg-gray-800">✉️ Email Roadmap</button>
                            </div>
                        </div>
                    </div>
                `);

                // Attach Event Listeners safely
                const actionBlock = document.getElementById(blockId);

                actionBlock.querySelector('.copy-btn').addEventListener('click', function() {
                    navigator.clipboard.writeText(data.response);
                    const original = this.innerHTML;
                    this.innerHTML = '✅ Copied!';
                    setTimeout(() => this.innerHTML = original, 2000);
                });

                actionBlock.querySelector('.export-btn').addEventListener('click', function() {
                    const blob = new Blob([data.response], { type: 'text/markdown' });
                    const url = URL.createObjectURL(blob);
                    const a = document.createElement('a');
                    a.href = url;
                    a.download = 'darby_roadmap.md';
                    a.click();
                    URL.revokeObjectURL(url);
                });

                actionBlock.querySelector('.email-btn').addEventListener('click', async function() {
                    const email = prompt("Enter your email address to receive this roadmap:");
                    if (!email) return;
                    const original = this.innerHTML;
                    this.innerHTML = 'Sending...';
                    try {
                        const res = await fetch('/api/email-roadmap', {
                            method: 'POST',
                            headers: { 'Content-Type': 'application/json' },
                            body: JSON.stringify({ email: email, markdown: data.response })
                        });
                        this.innerHTML = res.ok ? '✅ Sent!' : '❌ Failed';
                    } catch (e) {
                        this.innerHTML = '❌ Error';
                    }
                    setTimeout(() => this.innerHTML = original, 3000);
                });

                updateStats();

            } catch (err) {
                if(document.getElementById(loadingId)) document.getElementById(loadingId).remove();
                chatContainer.insertAdjacentHTML('beforeend', `<div class="text-red-500 text-sm pl-12">Error connecting to server.</div>`);
            }

            userInput.disabled = false;
            sendBtn.disabled = false;
            sendBtn.innerHTML = 'Send';
            userInput.focus();
            scrollToBottom();
        }

        function scrollToBottom() {
            window.scrollTo({ top: document.body.scrollHeight, behavior: 'smooth' });
        }
    </script>
</body>
</html>
"""

@app.get("/", response_class=HTMLResponse)
def serve_ui():
    return HTML_CONTENT

# 5. Connect to Ngrok and Run Server
nest_asyncio.apply()

# Ensure ngrok auth token is available, if not ask for it securely
if 'NGROK_AUTHTOKEN' not in globals() or not NGROK_AUTHTOKEN or not NGROK_AUTHTOKEN.startswith("ak_"):
    print("\n⚠️ Ngrok Auth Token is missing or invalid.")
    NGROK_AUTHTOKEN = getpass.getpass("Please paste your Ngrok Auth Token (Starts with 'ak_'): ")

try:
    ngrok.set_auth_token(NGROK_AUTHTOKEN)
    # Disconnect any existing tunnels to prevent conflicts
    ngrok.kill()

    # Open a ngrok tunnel on port 8000
    public_url = ngrok.connect(8000).public_url
    print("=" * 60)
    print(f"🚀 DARBY IS LIVE!")
    print(f"🌍 CLICK HERE TO OPEN THE UI: {public_url}")
    print("=" * 60)
    print("Note: Leave this cell running. To stop the server, click the stop button on this cell.")

    # Start Uvicorn Server
    uvicorn.run(app, host="0.0.0.0", port=8000)

except Exception as e:
    print(f"\n❌ Error starting Ngrok or Server: {e}")


⚠️ Ngrok Auth Token is missing or invalid.
Please paste your Ngrok Auth Token (Starts with 'ak_'): ··········
🚀 DARBY IS LIVE!
🌍 CLICK HERE TO OPEN THE UI: https://quotation-discard-debtor.ngrok-free.dev
Note: Leave this cell running. To stop the server, click the stop button on this cell.

❌ Error starting Ngrok or Server: asyncio.run() cannot be called from a running event loop


/tmp/ipykernel_1957/2400796165.py:291: RuntimeWarning: coroutine 'Server.serve' was never awaited
  print(f"\n❌ Error starting Ngrok or Server: {e}")
